🌐 [English](paddleocr_v6.ipynb) | [한국어](paddleocr_v6.ko.ipynb) | **日本語** | [中文](paddleocr_v6.zh.ipynb)

# DEEPX Tutorial 10 - DEEPX NPU 上の PP-OCRv6

このチュートリアルでは PP-OCRv6 を紹介し、DEEPX NPU 上での ONNX から DXNN までの完全な OCR ワークフローを実演します。コンパイルと授業での作業時間を短く保てるため、対象ティアとして PP-OCRv6_tiny を選択しています。

ワークフローは、個別に確認できる次のステージに分かれています。

1. 元の ONNX モデルをダウンロードする
2. 動的な入力次元を固定形状に置き換える
3. 固定形状のモデルが ONNX の結果を維持していることを検証する
4. モデルを DXNN にコンパイルする
5. 検出とテキスト認識を 1 つのアプリケーションとして実行する

## 学習目標

このチュートリアルを終えると、次のことができるようになります。

- PP-OCRv6 のアーキテクチャを説明し、tiny、small、medium のティアから選択する
- 検出 → 認識の OCR パイプラインを説明する
- 1 つの動的な認識モデルが 6 つの固定形状 DXNN モデルになる理由を説明する
- 元の ONNX ファイルを安全にダウンロードして検証する
- 前処理がアプリケーションと一致するキャリブレーション設定を作成する
- 失敗を隠さずに <code>dxcom</code> でモデルをコンパイルする
- 生成された DXNN 成果物を検査する
- レビュー済みのカメラアプリケーションを実行し、実用上の OCR 精度の限界を把握する

## このアプリケーションの NPU の使い方

| 項目 | このチュートリアルでは | 学んだ場所 |
|---|---|---|
| エンジン | 6 つの `InferenceEngine` オブジェクトが常駐します。検出器 1 つと認識器 5 つで、テキストの切り出しごとにアスペクト比で選択されます | T06-2 §3 |
| 実行 | 検出は同期的に実行されます(`run()`)。認識器はフレーム内の各切り出しを `run_async()` で実行し、`wait()` で結果を収集します | T06-2 §4, §5 |
| タスクグラフ | コンパイル済みの各モデルには NPU タスクとそれに続く `cpu_0` タスクがあります(DX-COM が CPU に残した演算子を ONNX Runtime が実行します) | T06-1 §6 |
| リクエストあたりの入力 | DET `[1, 640, 640, 3]` UINT8 = 1.2 MB。REC の切り出しは高さ 48 ピクセル、幅 120〜1200 | T06-3 §4 |
| 測定するもの | 検出器と認識器 1 つの `dxrun` スループット(6.3 節)をアプリケーションのフレームあたり時間と比較します。単語の多いフレームは REC リクエストも多くなるため、レイテンシはテキスト数に対して報告します | T06-1 §5 |

## 前提条件

- チュートリアル 01 が完了していること: DX-COM、DEEPX NPU を備えた DX-RT、および `uv`。
- ダウンロード: Hugging Face からの ONNX モデル 2 つ(1.8 MB と 4.5 MB)と、`cs.deepx.ai` からのキャリブレーションアーカイブ(110 MB)。
- 所要時間: 約 15 分。第 6 節の 6 回の `dxcom` コンパイルはそれぞれ数秒(検出器は約 10 秒)です。7.1 節では `workspace/.venv-ocr` 配下に 90 MB の Python 環境を作成します。
- DX-RT 3.4.2 と DX-COM 2.4.1 で検証済みです。モデルはここでコンパイルするため、常にお使いのコンパイラと一致します。
- 第 8 節のカメラアプリケーションには USB カメラとディスプレイが必要です。`--check` スモークテストはどちらがなくても実行できます。

次のセルは SDK を探し、これらの要件を確認してステータス表を表示します。`MISSING` と表示された項目には、それを提供する手順が併記されます。

## 1. PP-OCRv6 の概要

### 1.1 OCR とは?

**光学文字認識**(OCR)は、さまざまな種類の文書(スキャンした紙の文書、PDF ファイル、デジタルカメラで撮影した画像)を、編集や検索が可能なデータに変換する技術です。

AI に「目」を与えるものと考えてください。一般的に、次の 2 段階のパイプラインで動作します。
1. テキスト検出: 画像内のどこにテキストがあるかを特定します(その周りにボックスを描きます)。
2. テキスト認識: そのボックス内の文字が何であるかを読み取ります。

### 1.2 PP-OCRv6 とは?

PaddleOCR は、Baidu が PaddlePaddle フレームワークをベースに開発した超軽量のオープンソース OCR システムです。

[PP-OCRv6](https://github.com/PaddlePaddle/PaddleOCR) は、PaddleOCR の汎用 OCR モデルファミリーの最新世代です。テキスト検出とテキスト認識の両方に新しい **PPLCNetV4** バックボーンを使用し、エッジデバイスからサーバーまでをカバーする 3 つのデプロイティアを提供します。

[公式の PP-OCRv6 技術ドキュメント](https://www.paddleocr.ai/latest/en/version3.x/algorithm/PP-OCRv6/PP-OCRv6.html)に記載されている主な改善点は次のとおりです。

- **スケーラブルな単一モデルファミリー:** tiny、small、medium のティアは 1.5M から 34.5M パラメータまでをカバーします
- **統一された多言語認識:** small と medium は 1 つのモデルで 50 言語をサポートし、tiny は 49 言語をサポートします(日本語は含まれません)
- **テキスト検出の改善:** RepLKFPN がネックのパラメータを削減しつつ受容野を拡大します
- **テキスト認識の改善:** EncoderWithLightSVTR が局所的なコンテキストと大域的なアテンションを組み合わせ、CTC が効率的な並列デコードを提供します
- **特殊シーンのカバー:** 公式の評価には、手書き文字、回転した文字やアート文字、デジタル表示、ドットマトリクス文字、タイヤの刻印、その他の産業用テキストが含まれます

<img src="assets/ppocrv6-backbone.jpg" style="max-width: 1000px; width: 100%;" alt="PP-OCRv6 の認識と検出のための PPLCNetV4 バックボーン設計">

*PPLCNetV4 はタスクに適応したダウンサンプリングを使用します。認識では水平方向のシーケンス情報を保持し、検出ではマルチスケールの特徴マップを生成します。出典: [公式 PaddleOCR PP-OCRv6 ドキュメント](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.md)。*

### 1.3 tiny、small、medium

各ティアは同じ PP-OCRv6 設計を異なるスケールで使用します。一般に、大きなティアほど難しいケースでの精度は向上しますが、モデルサイズと計算コストが増加します。

| ティア | パラメータ数 | 想定ターゲット | 検出 Hmean (%) | 認識精度 (%) | Intel Xeon OpenVINO (秒/画像) | NVIDIA A100 PaddlePaddle (秒/画像) |
|---|---:|---|---:|---:|---:|---:|
| **tiny** | 1.5M | エッジ / IoT | 80.6 | 73.5 | **0.20** | **0.13** |
| **small** | 7.7M | モバイル / デスクトップ | 84.1 | 81.3 | 0.59 | 0.25 |
| **medium** | 34.5M | サーバー / 最高精度 | **86.2** | **83.2** | 1.40 | 0.29 |

> **表の読み方:** 精度の値は PaddleOCR 内部のマルチシナリオベンチマークによるものです。速度は、一般画像と文書画像 200 枚に対する画像 1 枚あたりのエンドツーエンドの秒数で、画像 I/O、前処理、後処理、推論を含みます。これらの CPU/GPU の数値はティア間の相対的なトレードオフを説明するものであり、**DEEPX NPU の測定値ではなく**、DX-COM のコンパイル時間を予測するものでもありません。

<img src="assets/ppocrv6-performance.png" style="max-width: 1100px; width: 100%;" alt="公式の PP-OCRv6 検出・認識精度の比較">

*左: テキスト検出の平均 Hmean。右: テキスト認識の加重平均精度。出典: [公式 PaddleOCR PP-OCRv6 ドキュメント](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.md)。*

### 1.4 このチュートリアルが PP-OCRv6_tiny を選ぶ理由

このチュートリアルでは、授業中のモデル準備とコンパイル時間を最小限に抑えるために **tiny** ティアを選択しています。1.5M パラメータという小ささは、エッジや IoT へのデプロイの自然な出発点にもなります。これはチュートリアルの時間を考慮した判断であり、tiny が最良の本番モデルであると主張するものではありません。測定された精度の向上が追加リソースに見合う場合は、small または medium を選択してください。

tiny には 2 つの重要なトレードオフがあります。

1. 公式の検出精度と認識精度が small および medium より低い
2. 49 言語をサポートし日本語は含まれないのに対し、small と medium は 50 言語をサポートする

> 第 4 節で公式の PP-OCRv6_tiny 検出器と認識器の ONNX モデルをダウンロードします。続く 7.3 節では、ライブデモの前に、アプリケーションの認識辞書、前処理、テンソルコントラクトがこれらのモデルと一致していることを確認します。

<img src="assets/ppocrv6-detection-comparison.jpg" style="max-width: 1000px; width: 100%;" alt="産業用および難しいテキストに対する公式の PP-OCRv6 medium テキスト検出比較">

*この公式の定性的な図は PP-OCRv6_medium を使用しており、難しい検出シナリオを示しています。tiny ティアの精度結果ではありません。出典: [公式 PaddleOCR PP-OCRv6 ドキュメント](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.md)。*

## 2. チュートリアル環境の準備

このノートブックは、共有のチュートリアルパスヘルパーを通じて <code>config.json</code> を読み込みます。SDK がチュートリアルリポジトリの中にインストールされていることは前提としません。

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import os, shlex, shutil, sys

paths = setup_tutorial(needs=["dx_com", "dx_rt", "npu", "uv"])
paths.require("dx_com", "dx_rt", "uv")

# Files created by this tutorial stay under notebooks/T10-demo-paddleocr/workspace (ignored by git).
MODEL_DIR = WORKSPACE_DIR / "models"
CONFIG_DIR = WORKSPACE_DIR / "configs_v6"
OUTPUT_DIR = WORKSPACE_DIR / "outputs" / "paddleocr_v6"
for directory in (MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


print()
print(f"Workspace          : {WORKSPACE_DIR}")
print(f"Model directory    : {MODEL_DIR}")
print(f"Config directory   : {CONFIG_DIR}")
print(f"Output directory   : {OUTPUT_DIR}")


### 2.1 現在の uv 環境への Python パッケージのインストール

Jupyter 環境は uv で作成されており、<code>pip</code> モジュールが含まれていない場合があります。そのため次のセルでは、<code>%pip</code> の代わりに <code>uv pip install --python ...</code> を使用します。

セルの再実行は安全です。uv は要件をすでに満たしているパッケージを再利用します。

In [ ]:
# Packages used by this notebook to export, simplify, and check ONNX models.
# They are installed into the Jupyter kernel's environment; DX-COM keeps its own environment.
!uv pip install --python "{sys.executable}" --quiet onnx onnxruntime onnxsim opencv-python pillow


## 3. OCR パイプラインを理解する

PP-OCRv6 は 2 つのモデルステージで動作します。検出してから認識するという同じフローが、すべてのティア(tiny、small、medium)に適用されます。

| ステージ | 入力 | 出力 | 目的 |
|---|---|---|---|
| テキスト検出 | 画像全体、640 × 640 | テキストのポリゴン | テキスト領域を見つける |
| テキスト認識 | テキストの切り出し 1 つ、高さ 48 | 文字列 | ピクセルをテキストに変換する |

<img src="assets/ocr-workflow.jpg" style="max-width: 980px;" alt="PP-OCR ワークフロー">

PaddleOCR を DX NPU に適用するには、次の 4 つのステップが必要です。

1. PaddleOCR の ONNX モデルをダウンロードする

2. 動的な入力形状を固定する

3. DX NPU 向けに ONNX を *.dxnn にコンパイルする

4. DEEPX-SDK で OCR アプリケーションを実装する

## 4. リソースのダウンロードと検査

### 4.1 ONNX モデルのダウンロード

このチュートリアルでは、Hugging Face 上の公式 PaddlePaddle ONNX リポジトリから **tiny** の検出器と認識器を直接ダウンロードします。

- [PP-OCRv6_tiny_det_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_tiny_det_onnx)
- [PP-OCRv6_tiny_rec_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_tiny_rec_onnx)

ダウンロードするのは **DET** と **REC** の ONNX モデルのみです。任意のテキスト行方向分類器は、このチュートリアルでは使用しません。

チュートリアルの結果を再現可能にするため、各 URL は公式リポジトリの特定のリビジョンに固定されています。ダウンロードは HTTPS 証明書を検証し、一時的な <code>.part</code> ファイルに書き込み、SHA-256 チェックサムを検証してから保存先を置き換えます。一致する既存ファイルは再利用され、古いファイルや異なるファイルは再度ダウンロードされます。

In [ ]:
import hashlib
from urllib.request import Request, urlopen

MODEL_SOURCES = {
    "det.onnx": {
        "repository": "PaddlePaddle/PP-OCRv6_tiny_det_onnx",
        "revision": "2ba1506c0380b8f0b03dd142459aac66d4421f6c",
        "sha256": "193bab7a04fca699a6c82e6abb5b81bdb28177f0abd4062552b04908dafb19f8",
    },
    "rec.onnx": {
        "repository": "PaddlePaddle/PP-OCRv6_tiny_rec_onnx",
        "revision": "2612ab37152ae0a677521bae4e1e3d4fb4cf7c30",
        "sha256": "9ef676d6ed3c88256a2d92c640c44f25b0c40947e111b14b8be8f594091563e6",
    },
}

def sha256sum(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as model_file:
        for chunk in iter(lambda: model_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_model(source: dict, destination: Path) -> Path:
    expected_sha256 = source["sha256"]
    if destination.is_file() and sha256sum(destination) == expected_sha256:
        print(f"SKIP: {destination.name} is already the verified PP-OCRv6 tiny model "
              f"({destination.stat().st_size / 1_000_000:.1f} MB)")
        return destination
    if destination.exists():
        print(f"REPLACE: {destination.name} does not match the selected tiny model")

    url = (
        f"https://huggingface.co/{source['repository']}/resolve/"
        f"{source['revision']}/inference.onnx"
    )

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)

    request = Request(url, headers={"User-Agent": "dx-tutorials"})
    try:
        with urlopen(request, timeout=120) as response, temporary.open("wb") as output:
            expected = response.headers.get("Content-Length")
            shutil.copyfileobj(response, output)
        if temporary.stat().st_size == 0:
            raise RuntimeError(f"Downloaded file is empty: {destination.name}")
        if expected and temporary.stat().st_size != int(expected):
            raise RuntimeError(
                f"Incomplete download for {destination.name}: "
                f"{temporary.stat().st_size} of {expected} bytes"
            )
        actual_sha256 = sha256sum(temporary)
        if actual_sha256 != expected_sha256:
            raise RuntimeError(
                f"Checksum mismatch for {destination.name}: {actual_sha256}"
            )
        temporary.replace(destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(f"DOWNLOADED: {destination.name} "
          f"({destination.stat().st_size / 1_000_000:.1f} MB)")
    return destination

ONNX_MODELS = {
    name: download_model(source, MODEL_DIR / name)
    for name, source in MODEL_SOURCES.items()
}

### 4.2 キャリブレーションデータセットのダウンロード

OCR のキャリブレーション画像をダウンロードして、このチュートリアルディレクトリに展開します。アーカイブは次のディレクトリを作成します。

```text
models/
├── det_dataset/
└── rec_dataset/
```

アーカイブファイルが完了マーカーの役割を果たします。<code>ocr-dataset.tar.gz</code> がすでに存在する場合、ダウンロードと展開の両方がスキップされます。

In [ ]:
CALIBRATION_DATASET_URL = "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/ocr-dataset.tar.gz"
CALIBRATION_ARCHIVE = WORKSPACE_DIR / "ocr-dataset.tar.gz"   # about 110 MB

if CALIBRATION_ARCHIVE.is_file():
    print(f"SKIP: {CALIBRATION_ARCHIVE.name} already exists; download and extraction are not required.")
else:
    # Download to a .part file first, extract into the workspace, then keep the archive so the cell can skip next time.
    !wget --https-only -O "{CALIBRATION_ARCHIVE}.part" "{CALIBRATION_DATASET_URL}" \
        && tar -xzf "{CALIBRATION_ARCHIVE}.part" -C "{WORKSPACE_DIR}" \
        && mv "{CALIBRATION_ARCHIVE}.part" "{CALIBRATION_ARCHIVE}"

for dataset_dir in (MODEL_DIR / "det_dataset", MODEL_DIR / "rec_dataset"):
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Expected dataset directory was not found: {dataset_dir}. "
            f"Remove {CALIBRATION_ARCHIVE} and run this cell again."
        )
print(f"Calibration datasets: {MODEL_DIR / 'det_dataset'}, {MODEL_DIR / 'rec_dataset'}")


In [ ]:
import onnx

def tensor_shape(value_info):
    return [
        dim.dim_value if dim.HasField("dim_value") else dim.dim_param or "dynamic"
        for dim in value_info.type.tensor_type.shape.dim
    ]

for name, model_path in ONNX_MODELS.items():
    onnx.checker.check_model(str(model_path))
    model = onnx.load(model_path, load_external_data=False)
    inputs = [(value.name, tensor_shape(value)) for value in model.graph.input]
    outputs = [(value.name, tensor_shape(value)) for value in model.graph.output]
    print(f"{name:10} input={inputs} output={outputs} nodes={len(model.graph.node)}")

元のモデルでは、バッチサイズと画像の寸法が動的になっています。

動的形状は汎用の ONNX Runtime アプリケーションには便利ですが、DEEPX のコンパイルでは毎回、具体的な入力形状が必要です。

## 5. 動的な入力形状の固定

次の表が、固定形状モデルに関する唯一の信頼できる情報源です。形状は NCHW 順(バッチ、チャンネル、高さ、幅)です。

In [ ]:
FIXED_MODEL_SPECS = [
    {"name": "det_fixed",            "source": "det.onnx", "shape": [1, 3, 640, 640]},
    {"name": "rec_fixed_ratio_2_5",  "source": "rec.onnx", "shape": [1, 3, 48, 120]},
    {"name": "rec_fixed_ratio_5",    "source": "rec.onnx", "shape": [1, 3, 48, 240]},
    {"name": "rec_fixed_ratio_10",   "source": "rec.onnx", "shape": [1, 3, 48, 480]},
    {"name": "rec_fixed_ratio_15",   "source": "rec.onnx", "shape": [1, 3, 48, 720]},
    {"name": "rec_fixed_ratio_25",   "source": "rec.onnx", "shape": [1, 3, 48, 1200]}
]

for spec in FIXED_MODEL_SPECS:
    print(f"{spec['name']:22} {spec['shape']}")

### 5.1 テキスト認識モデルの入力形状を固定する

**なぜ 5 つの異なるモデルが必要なのか?**

NPU は固定の入力形状を必要とするため、短い単語('Hi' など)と長い文を同じ幅 1200 のボックスで扱うと、過剰なパディングが発生し、細部が失われてしまいます。

異なるアスペクト比の「バケット」を作ることで、テキストをより効率的かつ正確に処理できます。そのため、このチュートリアルではアスペクト比の異なる 5 つの独立した認識モデルを使用します(検出モデル 1 つを加えて、合計 6 つの DXNN ファイル)。

各ケースで、検出されたテキストの比率に最も合うモデルを選択して適用します。

<img src="assets/ocr-ratio.png" style="max-width: 800px;">

次の GIF アニメーションは、実際に検出されたテキストの比率に基づいてどのテキスト認識モデルが選ばれるかを示しています。

<img src="assets/ocr-ratio.gif" style="max-width: 800px;">

### 5.2 `onnxsim` を使って動的な入力形状を固定する

ノートブックは、現在のカーネルの Python インタープリターを通じて ONNX Simplifier を呼び出します。たとえば、最初の変換は次と同等です。

~~~bash
python -m onnxsim models/det.onnx models/det_fixed.onnx           --overwrite-input-shape x:1,3,640,640
python -m onnxsim models/rec.onnx models/rec_fixed_ratio_2_5.onnx --overwrite-input-shape x:1,3,48,120
python -m onnxsim models/rec.onnx models/rec_fixed_ratio_5.onnx   --overwrite-input-shape x:1,3,48,240
...
~~~

空でない既存の固定形状モデルは確認のうえ再利用されます。

In [ ]:
def fixed_model_path(spec) -> Path:
    return MODEL_DIR / f"{spec['name']}.onnx"

for spec in FIXED_MODEL_SPECS:
    source = MODEL_DIR / spec["source"]
    destination = fixed_model_path(spec)

    if destination.is_file() and destination.stat().st_size > 0:
        try:
            onnx.checker.check_model(str(destination))
            print(f"SKIP: {destination.name} already exists and is valid")
            continue
        except Exception:
            print(f"REBUILD: {destination.name} is not a valid ONNX model")
            destination.unlink()

    shape_argument = "x:" + ",".join(map(str, spec["shape"]))
    print(f"\nonnxsim {source.name} -> {destination.name} ({shape_argument})")
    !"{sys.executable}" -m onnxsim "{source}" "{destination}" --overwrite-input-shape {shape_argument}
    onnx.checker.check_model(str(destination))

### 5.3 形状と数値的等価性の検証

構造的な検証だけでは十分ではありません。次のセルでは以下を行います。

1. 固定形状のすべての ONNX モデルを確認する
2. 正確な入力形状を確認する
3. 検出と代表的な 2 つの認識バケットについて ONNX Runtime の出力を比較する

すべての認識バケットは同じ元のグラフから生成されています。チュートリアルの検証を手早く済ませるため、数値的な認識チェックには比率 2.5 と比率 5 のモデルを使用します。

In [ ]:
for spec in FIXED_MODEL_SPECS:
    path = fixed_model_path(spec)
    onnx.checker.check_model(str(path))
    model = onnx.load(path, load_external_data=False)
    actual_shape = tensor_shape(model.graph.input[0])
    if actual_shape != spec["shape"]:
        raise AssertionError(
            f"{path.name}: expected {spec['shape']}, got {actual_shape}"
        )
    print(f"PASS: {path.name:28} input={actual_shape}")

In [ ]:
import numpy as np
import onnxruntime as ort

NUMERICAL_CHECKS = [
    FIXED_MODEL_SPECS[0],  # detection
    FIXED_MODEL_SPECS[1],  # recognition ratio 2.5
    FIXED_MODEL_SPECS[2],  # recognition ratio 5
]

rng = np.random.default_rng(2026)

for spec in NUMERICAL_CHECKS:
    source_path = MODEL_DIR / spec["source"]
    fixed_path = fixed_model_path(spec)
    sample = rng.random(spec["shape"], dtype=np.float32)

    source_session = ort.InferenceSession(
        str(source_path), providers=["CPUExecutionProvider"]
    )
    fixed_session = ort.InferenceSession(
        str(fixed_path), providers=["CPUExecutionProvider"]
    )
    source_output = source_session.run(None, {"x": sample})
    fixed_output = fixed_session.run(None, {"x": sample})

    if len(source_output) != len(fixed_output):
        raise AssertionError(f"Output count changed for {spec['name']}")

    max_error = max(
        float(np.max(np.abs(reference - candidate)))
        for reference, candidate in zip(source_output, fixed_output)
    )
    equivalent = all(
        np.allclose(reference, candidate, rtol=1e-4, atol=1e-5)
        for reference, candidate in zip(source_output, fixed_output)
    )
    print(f"{spec['name']:22} equivalent={equivalent} max_abs_error={max_error:.3e}")
    if not equivalent:
        raise AssertionError(f"Numerical output changed for {spec['name']}")

## 6. ONNX を DX NPU 向けの *.dxnn にコンパイルする

### 6.1 DX-COM のキャリブレーション設定を作成する

キャリブレーション時の前処理はアプリケーションの前処理と一致していなければなりません。色順、スケーリング、正規化、レイアウトのいずれかが食い違うと、コンパイルが成功しても精度が低下することがあります。

| モデル | キャリブレーション画像 | リサイズ | 平均 / 標準偏差 |
|---|---|---:|---|
| 検出 | <code>det_dataset</code> | 640 × 640 | ImageNet の値 |
| 認識 | 対応するアスペクト比のバケット | 固定幅 × 48 | [0.5, 0.5, 0.5] |

In [ ]:
import json

# The archive from 4.2 was extracted into the workspace (workspace/models/...).
DET_DATASET = MODEL_DIR / "det_dataset"
REC_DATASET = MODEL_DIR / "rec_dataset"

required_datasets = [
    DET_DATASET,
    REC_DATASET / "ratio_5",
    REC_DATASET / "ratio_15",
    REC_DATASET / "ratio_25",
]
for dataset in required_datasets:
    if not dataset.is_dir():
        raise FileNotFoundError(f"Calibration dataset was not found: {dataset}")

def image_count(directory: Path) -> int:
    extensions = {".jpeg", ".jpg", ".png"}
    return sum(
        path.is_file() and path.suffix.lower() in extensions
        for path in directory.iterdir()
    )

for dataset in required_datasets:
    count = image_count(dataset)
    if count == 0:
        raise RuntimeError(f"No calibration images were found in {dataset}")
    print(f"{dataset.relative_to(WORKSPACE_DIR)!s:28} {count:4} images")

In [ ]:
def preprocessing(width, height, mean, std):
    return [
        {"resize": {"width": width, "height": height}},
        {"convertColor": {"form": "BGR2RGB"}},
        {"div": {"x": 255}},
        {"normalize": {"mean": mean, "std": std}},
        {"transpose": {"axis": [2, 0, 1]}},
        {"expandDim": {"axis": 0}},
    ]

def calibration_config(shape, dataset, calibration_num, mean, std):
    _, _, height, width = shape
    return {
        "inputs": {"x": shape},
        "calibration_num": calibration_num,
        "calibration_method": "ema",
        "default_loader": {
            "dataset_path": str(dataset),
            "file_extensions": ["jpeg", "jpg", "png", "JPEG"],
            "preprocessings": preprocessing(width, height, mean, std),
        }
    }

REC_DATASET_BY_BUCKET = {
    3: REC_DATASET / "ratio_5",
    5: REC_DATASET / "ratio_5",
    10: REC_DATASET / "ratio_15",
    15: REC_DATASET / "ratio_15",
    25: REC_DATASET / "ratio_25",
    35: REC_DATASET / "ratio_25",
}

COMPILE_SPECS = []

for spec in FIXED_MODEL_SPECS:
    if spec["name"] == "det_fixed":
        config = calibration_config(
            spec["shape"], DET_DATASET, 100,
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225],
        )
    else:
        bucket = int(spec["name"].rsplit("_", 1)[1])
        config = calibration_config(
            spec["shape"], REC_DATASET_BY_BUCKET[bucket], 80,
            [0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5],
        )

    config_path = CONFIG_DIR / f"{spec['name']}.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    COMPILE_SPECS.append({
        **spec,
        "onnx": fixed_model_path(spec),
        "config": config_path,
        "output_dir": OUTPUT_DIR / spec["name"],
    })
    print(f"WROTE: {config_path.relative_to(TUTORIAL_DIR)}")

コンパイルの前に、少なくとも 1 つの設定を確認してください。特に NCHW の入力形状、キャリブレーションデータセット、リサイズサイズ、チャネル順、正規化、transpose の順序を確認します。

In [ ]:
example_config = COMPILE_SPECS[0]["config"]
print(example_config.read_text(encoding="utf-8"))

### 6.2 ONNX モデルを DXNN にコンパイルする

各モデルはそれぞれ専用のディレクトリに書き出されます。有効な DXNN ファイルがすでに存在する場合はスキップされるため、授業を繰り返しても再コンパイルに時間を取られません。

セルはモデル名を表示したうえで、各モデルに対して正確に次のコマンドを実行します。

~~~bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m <fixed-model.onnx> \
      -c <calibration-config.json> \
      -o <output-directory> \
      --gen_log \
      --export_html
~~~

コンパイラの出力はそのまま表示され、コンパイルに失敗した場合は黙って続行せずにエラーでセルが停止します。

In [ ]:
# Keep all entries for a complete OCR application.
# During a short class, you may select a smaller list only to demonstrate compilation.
SELECTED_COMPILES = [spec["name"] for spec in COMPILE_SPECS]
print("Models selected for compilation:")
for name in SELECTED_COMPILES:
    print(" -", name)

In [ ]:
def expected_dxnn(spec) -> Path:
    return spec["output_dir"] / f"{spec['name']}.dxnn"

def compile_model(spec) -> Path:
    artifact = expected_dxnn(spec)
    if artifact.is_file() and artifact.stat().st_size > 0:
        print(f"SKIP: {artifact.relative_to(TUTORIAL_DIR)} already exists")
        return artifact

    spec["output_dir"].mkdir(parents=True, exist_ok=True)
    onnx_path, config_path, output_dir = spec["onnx"], spec["config"], spec["output_dir"]
    print(f"\n===== dxcom: {spec['name']} =====")
    # Same command as in a terminal after `source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate`.
    !cd "{WORKSPACE_DIR}" && source "{DX_COMPILER_VENV}/bin/activate" && \
        dxcom -m "{onnx_path}" -c "{config_path}" -o "{output_dir}" --gen_log --export_html

    if not artifact.is_file() or artifact.stat().st_size == 0:
        raise FileNotFoundError(f"DX-COM did not create {artifact}")
    return artifact

COMPILED_MODELS = {}
for spec in COMPILE_SPECS:
    if spec["name"] in SELECTED_COMPILES:
        COMPILED_MODELS[spec["name"]] = compile_model(spec)

### 6.3 コンパイル成果物を検証する

完全な OCR アプリケーションには、検出 1 個と認識バケット 5 個、合計ちょうど 6 個の DXNN ファイルが必要です。必要なファイルが 1 つでも欠けていると、このセルは早い段階で失敗します。

In [ ]:
REQUIRED_DXNN_NAMES = [spec["name"] for spec in COMPILE_SPECS]
missing = []

for spec in COMPILE_SPECS:
    artifact = expected_dxnn(spec)
    if artifact.is_file() and artifact.stat().st_size > 0:
        print(f"PASS: {artifact.relative_to(TUTORIAL_DIR)!s:55} "
              f"{artifact.stat().st_size / 1_000_000:7.2f} MB")
    else:
        missing.append(artifact)

if missing:
    raise FileNotFoundError(
        "Missing DXNN files:\n" + "\n".join(f" - {path}" for path in missing)
    )

<code>dxparse</code> で 2 つのモデルの役割を確認します。認識モデルはアスペクト比 2.5 のものが代表例で、他の認識ファイルとの違いは主に固定入力幅だけです。

In [ ]:
for model_name in ("det_fixed", "rec_fixed_ratio_2_5"):
    model_path = next(expected_dxnn(spec) for spec in COMPILE_SPECS if spec["name"] == model_name)
    print("\n" + "=" * 72)
    !dxparse -m "{model_path}" -v


同じ 2 つのモデルについて CLI のベースラインを測定します。どちらのグラフも CPU タスクで終わるため `--use-ort` が必須で、`-v` を付けるとレイテンシの内訳が表示されます。これらの数値は後でアプリケーションと比較します。カメラループは検出した単語ごとに DET 1 回と REC 1 回のリクエストを実行するため、フレーム時間はテキスト量に応じて増えます。

```bash
dxrun -m <workspace>/outputs/paddleocr_v6/det_fixed/det_fixed.dxnn --use-ort -t 3 -v
dxrun -m <workspace>/outputs/paddleocr_v6/rec_fixed_ratio_2_5/rec_fixed_ratio_2_5.dxnn --use-ort -t 3 -v
```

In [ ]:
for model_name in ("det_fixed", "rec_fixed_ratio_2_5"):
    model_path = next(expected_dxnn(spec) for spec in COMPILE_SPECS if spec["name"] == model_name)
    print(f"\n===== dxrun: {model_name} =====")
    !dxrun -m "{model_path}" --use-ort -t 3 -v 2>&1 | grep -E "Task\[|FPS|Latency|NPU Processing" 


## 7. DEEPX-SDK で OCR アプリケーションをテストする

再利用可能な Python アプリケーションは <code>app/</code> にあります。<code>outputs/paddleocr_v6/</code> にコンパイル済みの PP-OCRv6 tiny モデルを使い、各フレームを次のように処理します。

<img src="assets/ppocrv6-camera-pipeline.png" alt="PP-OCRv6 カメラ推論パイプライン" style="max-width: 1000px; width: 100%; height: auto;">

検出器と 5 つの認識エンジンは一度だけロードされ、常駐したままになります。

### 7.1 アプリケーションの依存関係をインストールする

アプリケーションには <code>dx_engine</code>、OpenCV、NumPy、Pillow が必要です。チュートリアル 06-2 と同様に、これらは共有の Jupyter 環境ではなく <code>workspace/.venv-ocr</code> 配下の小さな環境にインストールします。DX-RT パッケージは <code>/usr/share/libdxrt-bin/python</code> にビルド済みの <code>dx_engine</code> wheel を同梱しており、セルはカーネルの Python に合う <code>cpXY</code> タグの wheel を選びます。<code>app/run_camera.sh</code> も同じ環境を使います。セルの再実行は安全で、すでに存在するものはスキップされます。

```bash
uv venv --python <jupyter-python> <T10>/workspace/.venv-ocr
uv pip install --python <T10>/workspace/.venv-ocr/bin/python -r app/requirements.txt \
    /usr/share/libdxrt-bin/python/dx_engine-<version>-cp<XY>-*.whl
```

In [ ]:
APP_REQUIREMENTS = APP_DIR / "requirements.txt"
CAMERA_APP = APP_DIR / "camera_app.py"
for required in (APP_DIR, APP_REQUIREMENTS, CAMERA_APP):
    if not required.exists():
        raise FileNotFoundError(f"Required application path was not found: {required}")

# The prebuilt dx_engine wheel that matches this kernel's Python version.
WHEEL_DIR = Path("/usr/share/libdxrt-bin/python")
python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
matching_wheels = sorted(WHEEL_DIR.glob(f"dx_engine-*-{python_tag}-{python_tag}-*.whl"))
if not matching_wheels:
    raise FileNotFoundError(f"No dx_engine wheel for {python_tag} under {WHEEL_DIR}. Install the libdxrt-bin package (Tutorial 01 section 3).")
DX_ENGINE_WHEEL = matching_wheels[-1]

# Tutorial-local environment for the application (same approach as Tutorial 06-2).
OCR_PYTHON_ENV = WORKSPACE_DIR / ".venv-ocr"
OCR_PYTHON = OCR_PYTHON_ENV / "bin" / "python"
if not OCR_PYTHON.is_file():
    !uv venv --python "{sys.executable}" "{OCR_PYTHON_ENV}"
else:
    print(f"Environment already exists: {OCR_PYTHON_ENV}")
!uv pip install --python "{OCR_PYTHON}" --quiet -r "{APP_REQUIREMENTS}" "{DX_ENGINE_WHEEL}"

!"{OCR_PYTHON}" -c "import cv2, dx_engine; print('OpenCV   :', cv2.__version__); print('dx_engine:', dx_engine.__file__)"


### 7.2 アプリケーションの構成と安全策

この Notebook は、別のランナーを生成する代わりに <code>app/</code> 配下で保守されているファイルを使います。

| ファイル | 役割 |
|---|---|
| <code>app/ocr_engine.py</code> | DXNN のロード、DET の後処理、切り出し、5 バケットの REC ルーティング、CTC デコード |
| <code>app/camera_app.py</code> | CLI の検証、640×480 のカメラキャプチャ、OpenCV プレビュー、BBOX と多言語テキストのオーバーレイ |
| <code>app/run_camera.sh</code> | <code>workspace/.venv-ocr</code> でアプリケーションを起動(<code>PYTHON_BIN</code> で上書き可能) |
| <code>app/assets/ppocrv6_tiny_dict.txt</code> | 6906 クラスの REC 出力に対応する公式 tiny 辞書 |

モデルの欠落、無効なテンソル形状、辞書の不一致、カメラのオープン/読み取り失敗に対して、アプリケーションは明確に失敗します。既存の DXNN ファイルを変更することはありません。

### 7.3 カメラなしでアプリケーションのスモークテストを実行する

<code>--check</code> モードはカメラを開かず、GUI ウィンドウも作成しません。パッケージ化されたアプリケーションをロードし、検出器と 5 つの認識器すべてに合成入力の推論を 1 回通します。これにより、ライブデモの前に NPU ランタイム、入出力テンソルのコントラクト、tiny 辞書のクラス数を検証できます。

下で実行するコマンドは、モデルと辞書のパスを明示して <code>workspace/.venv-ocr/bin/python app/camera_app.py --check</code> を実行するのと同等です。

In [ ]:
APP_DICTIONARY = APP_DIR / "assets" / "ppocrv6_tiny_dict.txt"
REQUIRED_MODEL_NAMES = [
    "det_fixed",
    "rec_fixed_ratio_2_5",
    "rec_fixed_ratio_5",
    "rec_fixed_ratio_10",
    "rec_fixed_ratio_15",
    "rec_fixed_ratio_25",
]

required_files = [CAMERA_APP, APP_DICTIONARY]
required_files.extend(
    OUTPUT_DIR / name / f"{name}.dxnn"
    for name in REQUIRED_MODEL_NAMES
)
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Application test files are missing:\n"
        + "\n".join(f" - {path}" for path in missing_files)
    )

# Synthetic-input smoke test through all six models (no camera needed).
!cd "{WORKSPACE_DIR}" && "{OCR_PYTHON}" "{CAMERA_APP}" --check --model-dir "{OUTPUT_DIR}" --dictionary "{APP_DICTIONARY}"


### 7.4 640×480 のカメラアプリケーションをテストする

ライブアプリケーションは対話型の OpenCV ウィンドウを開きます。Notebook のカーネルをブロックせず、カメラを正しく解放できるように、**JupyterLab のターミナル**で実行してください。`run_camera.sh` は 7.1 で作成した `workspace/.venv-ocr` 環境で起動します。

既定の入力は <code>/dev/video0</code> の 640×480、15 FPS です。プレビューウィンドウで **q** または **Esc** を押すと終了します。別のデバイスを選ぶには <code>--camera /dev/videoN</code> を使います。

In [ ]:
camera_command = [
    str(APP_DIR / "run_camera.sh"),
    "--camera", "/dev/video0",
    "--width", "640",
    "--height", "480",
    "--fps", "15",
]

print("Run in a separate JupyterLab Terminal:")
print(f"cd {shlex.quote(str(APP_DIR))}")
print(shlex.join(["./run_camera.sh", *camera_command[1:]]))

<img src="assets/sc-ocr-app.png" style="max-width: 800px;" alt="期待される PP-OCR の結果">

## 8. 精度と性能のチェックリスト

コンパイルが成功しても、OCR の精度が十分であるとは限りません。代表的な画像でパイプライン全体を検証してください。

| 確認項目 | 重要な理由 | 実際の対応 |
|---|---|---|
| 検出解像度 | 小さな文字は 640 × 640 で消えることがある | 想定するカメラ距離で再現率を比較する |
| キャリブレーションのカバー範囲 | 量子化はキャリブレーション統計に従う | 実際の照明、フォント、ぼけ、背景を含める |
| 前処理の一致 | 色や正規化の不一致はモデル入力をずらす | コンパイル時と実行時の前処理を同一に保つ |
| 認識バケット | 過度なリサイズやパディングは文字を損なう | テキストのアスペクト比の分布を測定する |
| 認識の信頼度 | しきい値が低いと誤ったテキストが表示され、高いとテキストが落ちる | ラベル付き検証セットで調整する |
| 遠近と向き | カメラで撮った文書は常に平らで正立とは限らない | 必要に応じて文書の向き補正と歪み補正を追加する |
| エンドツーエンドのレイテンシ | 1 フレームに多数の認識用切り出しが含まれることがある | FPS だけでなくテキスト数に対するレイテンシを報告する |

見栄えのよいカメラサンプル 1 枚に合わせて調整しないでください。固定の検証セットを維持し、設定を変更するたびに検出の再現率、認識精度、エンドツーエンドのレイテンシを記録します。

## 9. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| `dxcom` がない | ステータス表に `[MISSING] dx_com` | DX-COM がインストールされていないか、`config.json` が別の場所を指している | チュートリアル 01 の第 2 節を完了し、`python tutorial_paths.py --show` を確認する |
| ダウンロードが拒否される | `Checksum mismatch` または `Incomplete download` | 転送が中断されたか、上流のファイルが変更された | `workspace/models` 配下の該当モデルだけを削除して 4.1 節を再実行する |
| 固定モデルの形状が違う | `AssertionError: ... expected [1, 3, 48, 240]` | 古い固定 ONNX ファイルが残っている | その固定 ONNX ファイルを削除して第 5 節を再実行する |
| コンパイルが失敗する | `Error Type: DataNotFoundError` などの `dxcom` の `[ERROR]` 行 | `dataset_path` が間違っているか、指定した拡張子の画像がない | `configs_v6/<model>.json` を 6.1 で表示されたディレクトリと比較する。完全なログは `outputs/paddleocr_v6/<model>/compiler.log` にある |
| 7.1 で `dx_engine` のインポートに失敗する | `ModuleNotFoundError: No module named 'dx_engine'` または wheel/ABI の不一致 | 別の Python で環境が作られたか、DX-RT がアップグレードされた | `workspace/.venv-ocr` を削除して 7.1 を再実行する。wheel はインストール済みの `libdxrt-bin` と一致している必要がある |
| `--check` がモデルの欠落を報告する | `Application test files are missing` | 6 個の DXNN ファイルがすべてコンパイルされていない | `SELECTED_COMPILES` にすべてのモデルを含めて 6.2 を再実行する |
| カメラを開けない | `Could not open camera /dev/video0` | カメラがない、デバイスパスが違う、またはユーザーが `video` グループに入っていない | `v4l2-ctl --list-devices` を実行し、`--camera /dev/videoN` を渡す。`sudo usermod -aG video $USER` の後に再ログインする |
| OpenCV の表示エラー | `cannot open display` | グラフィカルセッションがない | デスクトップセッション(有効な `DISPLAY` を持つ JupyterLab ターミナル)から実行する |
| テキストが表示されない | テキストのないボックス、またはボックス自体がない | 照明、フォーカス、しきい値 | 代表的な画像で `--det-threshold`、`--box-threshold`、`--rec-threshold` を調整する |

## 10. まとめ

### 10.1 完了したワークフロー

```text
Official PP-OCRv6 tiny DET + REC ONNX
                    │
                    ▼
       Fix dynamic input dimensions
          ┌─────────┴─────────┐
          │                   │
   DET 640×640       REC width buckets ×5
          └─────────┬─────────┘
                    ▼
     Calibration configuration + DX-COM
                    │
                    ▼
           Six verified DXNN models
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
 camera_app.py --check    640×480 camera
                              │
                              ▼
                   BBOX + recognized text
```

### 10.2 成果物と検証のダッシュボード

| 段階 | 成果物またはコマンド | 検証内容 |
|---|---|---|
| 元のモデル | <code>det.onnx</code>、<code>rec.onnx</code> | 公式リビジョンと SHA-256 チェックサム |
| 固定 ONNX | DET 1 個と REC 5 個のモデル | 静的な入力形状と ONNX の有効性 |
| 数値チェック | ONNX Runtime による比較 | 固定形状への変換が代表的な出力を保つこと |
| キャリブレーション | <code>configs_v6/*.json</code> | データセット、リサイズ、色順、正規化、レイアウト |
| コンパイル | <code>dxcom</code> | ログとレポートが見える状態での ONNX から DXNN への変換 |
| ランタイムコントラクト | <code>camera_app.py --check</code> | DET と 5 つの REC モデルが期待どおりのテンソルコントラクトで実行されること |
| ライブアプリケーション | <code>app/run_camera.sh</code> | 640×480 のキャプチャ、アスペクト比によるルーティング、CTC デコード、BBOX、テキストオーバーレイ |

### 10.3 ランタイムのモデルマップ

以下の形状は DX-RT が実行時に報告するもの(`dxparse`、`get_input_tensors_info()`)で、バッチ、高さ、幅、チャネルの順(NHWC、UINT8)です。第 5 節では ONNX の入力を NCHW の float 順で固定しました。DX-COM がレイアウト変換と正規化をコンパイル済みモデルに畳み込むため、アプリケーションは生の HWC ピクセルをそのまま入力できます。

| 役割 | 固定入力 | 選択ルール |
|---|---:|---|
| DET | `[1, 640, 640, 3]` | カメラフレームごとに 1 回 |
| REC 2.5 | `[1, 48, 120, 3]` | 切り出しのアスペクト比 ≤ 2.5 |
| REC 5 | `[1, 48, 240, 3]` | 切り出しのアスペクト比 ≤ 5 |
| REC 10 | `[1, 48, 480, 3]` | 切り出しのアスペクト比 ≤ 10 |
| REC 15 | `[1, 48, 720, 3]` | 切り出しのアスペクト比 ≤ 15 |
| REC 25 | `[1, 48, 1200, 3]` | 切り出しのアスペクト比 ≤ 25、またはそれより横長の切り出しのフォールバック |

### 10.4 完了チェックリスト

- [ ] 公式の PP-OCRv6 tiny DET と REC の ONNX モデルをダウンロードする
- [ ] 元のモデルのチェックサムと ONNX 構造を検証する
- [ ] 固定 DET モデル 1 個と固定 REC モデル 5 個を作成する
- [ ] 代表的な元モデルと固定モデルの出力を比較する
- [ ] キャリブレーション設定を作成し、DXNN モデルをコンパイルする
- [ ] コンパイル成果物を確認し、6 個のモデルすべてを NPU でテストする
- [ ] DET から REC への OpenCV カメラアプリケーションを準備する
- [ ] ラベル付き検証セットで検出精度と認識精度を測定する
- [ ] 代表的なカメラ画像でしきい値を調整する
- [ ] エンドツーエンドのレイテンシ、スループット、ホスト CPU 使用率を記録する

### 10.5 small または medium ティアを試す

より大きな PP-OCRv6 ティアで同じワークフローを繰り返すには、対応する公式 PaddlePaddle ONNX リポジトリから検出器と認識器の両方をダウンロードします。

| ティア | 検出 ONNX リポジトリ | 認識 ONNX リポジトリ |
|---|---|---|
| **small** | [PP-OCRv6_small_det_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_small_det_onnx) | [PP-OCRv6_small_rec_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_small_rec_onnx) |
| **medium** | [PP-OCRv6_medium_det_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_det_onnx) | [PP-OCRv6_medium_rec_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_rec_onnx) |

### 10.6 OCR の性能を向上させるには

このチュートリアルでは、`Document Image Orientation Classification`、`Text Image Unwarping`、`Text Line Orientation Classification` といった前処理ブロックは実装していません。

より高い OCR 性能が必要な場合は、不足している AI モデルを実装して OCR の AI パイプラインに組み込んでください。

<img src="assets/ppocrv6-full-pipeline.png" style="max-width: 1200px;">

精度を高めるには:

- **実際の運用環境**のテキスト画像でキャリブレーションデータセットを構築する。
- 実際のカメラ、距離、文字サイズ、アスペクト比、フォント、言語、照明、ぼけ、遠近をカバーする。
- 現在の**キャリブレーションデータセット**には、アスペクト比 5、15、25 の 3 つの大まかな REC バケットがある。運用時のテキスト形状のばらつきが大きい場合は、これらのデータセットのバケットをより細かく分割する。
- これらのキャリブレーション用バケットは、実行時の 5 つの REC モデルルート(アスペクト比 2.5、5、10、15、25)とは別のものである。
- 動的な DET と REC の入力を、運用データの測定結果に基づいて選んだ固定形状に変換する。
- キャリブレーションと実行時の前処理を同一に保つ。

| 運用データ | 調整 |
|---|---|
| 現在のバケットの中間にあるアスペクト比 | 中間の REC バケットを追加する |
| 縦長、細い、または長いテキスト | 対応する固定 REC 形状を使う |
| 非常に小さい文字 | より大きな DET 入力を評価する |
| 特殊なカメラのアスペクト比 | DET の入力形状を合わせる |
| リサイズ後の歪み | 形状と前処理を調整する |

リリース前に:

- 同じラベル付きデータセットで ONNX と DXNN の精度を比較する。
- 検出の再現率、認識精度、レイテンシ、メモリ、CPU 負荷を測定する。
- テンソルコントラクト、前処理、キャリブレーション、認識辞書を再確認する。
- 測定可能な価値がある場合にだけ形状を追加する。形状が増えるとビルド時間、ストレージ、ルーティングの複雑さが増える。

> **次へ:** チュートリアル 20 に進み、同じランタイム上でマルチチャネルの YOLO アプリケーションを C++ でビルドして実行しましょう。